In [28]:
import json
import os
import copy
from typing import Dict, List, Any, Optional, Tuple

def load_metadata_files(session_folder: str) -> Tuple[Optional[Dict[str, Any]], Optional[Dict[str, Any]]]:
    """
    Load session.json and rig.json files from a session folder.
    
    Args:

        session_folder: Path to the session directory
        
    Returns:
        Tuple of (session_metadata, rig_metadata) or (None, None) if files not found
    """
    metadata_path = os.path.join(session_folder, "metadata")
    
    if not os.path.exists(metadata_path):
        print(f"Metadata folder not found: {metadata_path}")
        return None, None
    
    session_json_path = os.path.join(metadata_path, "session.json")
    rig_json_path = os.path.join(metadata_path, "rig.json")
    
    session_metadata = None
    rig_metadata = None
    
    if os.path.exists(session_json_path):
        with open(session_json_path, 'r') as f:
            session_metadata = json.load(f)
        print(f"Loaded session.json from: {session_json_path}")
    else:
        print(f"session.json not found: {session_json_path}")
    
    if os.path.exists(rig_json_path):
        with open(rig_json_path, 'r') as f:
            rig_metadata = json.load(f)
        print(f"Loaded rig.json from: {rig_json_path}")
    else:
        print(f"rig.json not found: {rig_json_path}")
    
    return session_metadata, rig_metadata

def modify_session_json(session_metadata: Dict[str, Any]) -> Dict[str, Any]:
    """
    Apply all fixes to session.json metadata.
    
    Args:
        session_metadata: Original session metadata dictionary
        
    Returns:
        Modified session metadata with all fixes applied
    """
    # Create an unlinked copy of session_metadata 
    modified_session_json = copy.deepcopy(session_metadata)

    # Session['data_streams'] fixes 
    # merge the two behavior data streams 
    merged = deep_merge(session_metadata['data_streams'][0], session_metadata['data_streams'][1], 
                        preserve_keys={"stream_start_time", "stream_end_time"})
    # fix the camera names to match rig 
    merged['camera_names'] = ['Face forward', 'Body', 'Eye', 'Probe Camera']
    del merged['stim_modalities']
    merged['stream_modalities'].append({'name': 'Behavior', 'abbreviation': 'behavior'})

    # create new data stream for ecephys 
    ecephys_data_stream = {
        "stream_modalities": [{'name': 'Extracellular electrophysiology', 'abbreviation': 'ecephys'}],
        "stream_start_time": merged['stream_start_time'],
        "stream_end_time": merged['stream_end_time']
    }
    # update data streams object 
    modified_session_json['data_streams'] = [merged, ecephys_data_stream]

    # Session['stimulus_epochs'] fixes
    for epoch in modified_session_json['stimulus_epochs']: # fix parameters 
        del epoch['software'][0]['parameters']['taskScript']
        del epoch['software'][0]['parameters']['taskControl']
        del epoch['software'][0]['parameters']['taskUtils']
        del epoch['script']['parameters']['taskScript']
        del epoch['script']['parameters']['taskControl']
        del epoch['script']['parameters']['taskUtils']

        if epoch['stimulus_name'] == 'OptoTagging':
            epoch['stimulus_modalities'] = ['Optogenetics']
            # copy laser_duration, optotagging_locations_bregmaX, optotagging_locations_bregmaY, probes_targeted from the light_source_config object and move to software.parameters 
            if 'light_source_config' in epoch:
                epoch['software'][0]['parameters']['laser_duration'] = epoch['light_source_config'].get('laser_duration', None)
                epoch['software'][0]['parameters']['optotagging_locations_bregmaX'] = epoch['light_source_config'].get('optotagging_locations_bregmaX', None)
                epoch['software'][0]['parameters']['optotagging_locations_bregmaY'] = epoch['light_source_config'].get('optotagging_locations_bregmaY', None)
                epoch['software'][0]['parameters']['probes_targeted'] = epoch['light_source_config'].get('probes_targeted', None)
                # remove the copied fields from light_source_config 
                del epoch['light_source_config']['laser_duration']
                del epoch['light_source_config']['optotagging_locations_bregmaX']
                del epoch['light_source_config']['optotagging_locations_bregmaY']
                del epoch['light_source_config']['probes_targeted']

        if epoch['stimulus_name'] == 'RFMapping': 
            epoch['stimulus_parameters']['type'] = 'Sin'
            # change field names 
            epoch['stimulus_parameters']['trial_duration'] = epoch['stimulus_parameters'].pop('duration_sec', None)
            epoch['stimulus_parameters']['orientations'] = epoch['stimulus_parameters'].pop('orientations_deg', None)
            epoch['stimulus_parameters']['spatial_frequency_cycles'] = epoch['stimulus_parameters'].pop('spatial_frequency_cycles_per_deg', None)
            epoch['stimulus_parameters']['temporal_frequency_cycles'] = epoch['stimulus_parameters'].pop('temporal_frequency_cycles_per_sec', None)

            # add new fields for units 
            epoch['stimulus_parameters']['trial_duration_unit'] = 'seconds'
            epoch['stimulus_parameters']['orientations_unit'] = 'degrees'
            epoch['stimulus_parameters']['spatial_frequency_cycles_unit'] = 'cycles per degree'
            epoch['stimulus_parameters']['temporal_frequency_cycles_unit'] = 'seconds'

        if epoch['stimulus_name'] == 'Spontaneous':
            epoch['notes'] = 'low-luminance black screen'

    return modified_session_json

def deep_merge(obj1, obj2, preserve_keys=None):
    """ Merges two json objects, preserves specific keys and handles conflicts"""
    if preserve_keys is None:
        preserve_keys = set()

    result = {}

    keys = set(obj1) | set(obj2)
    for key in keys:
        val1 = obj1.get(key)
        val2 = obj2.get(key)

        if key in obj1 and key in obj2:
            if key in preserve_keys:
                # Assert values are equal or choose one
                if val1 != val2:
                    raise ValueError(f"Conflicting values for preserved key '{key}': {val1} vs {val2}")
                result[key] = val1
            elif isinstance(val1, dict) and isinstance(val2, dict):
                result[key] = deep_merge(val1, val2, preserve_keys)
            elif isinstance(val1, list) and isinstance(val2, list):
                # Customize list merge logic here (e.g., deduplication)
                result[key] = list(set(val1 + val2))
            else:
                # If both are scalars but not preserved, choose one or raise error
                result[key] = val2  # or raise ValueError or custom logic
        elif key in obj1:
            result[key] = val1
        else:
            result[key] = val2

    return result

def modify_rig_json(rig_metadata: Dict[str, Any]) -> Dict[str, Any]:
    """
    Apply all fixes to rig.json metadata.
    
    Args:
        rig_metadata: Original rig metadata dictionary
        
    Returns:
        Modified rig metadata with all fixes applied
    """
    # Create an unlinked copy of rig_metadata
    modified_rig_json = copy.deepcopy(rig_metadata)

    # Change laser name to match session.json 
    for light_source in modified_rig_json['light_sources']: 
        if light_source['device_type'] == 'Laser':
            if light_source['wavelength'] == 488: 
                light_source['name'] = 'laser_488'

            if light_source['wavelength'] == 633: 
                light_source['name'] = 'laser_633' 

    modified_rig_json['origin'] = 'Bregma'
    # Create 3 objects for the rig axes, copy from OpenScope metadata 
    modified_rig_json['rig_axes'] = {}
    modified_rig_json['rig_axes'][0] = {'direction': 'layers on the Mouse Sagittal Plane, Positive direction is towards the nose of the mouse',
                                        'name' : 'X'}
    modified_rig_json['rig_axes'][1] = {'direction': 'positive pointing UP opposite the direction from the force of gravity',
                                        'name' : 'Y'}
    modified_rig_json['rig_axes'][2] = {'direction': 'defined by the right hand rule and the other two axis',
                                        'name' : 'Z'}
    return modified_rig_json

def save_modified_metadata(session_name: str, modified_session: Dict[str, Any], 
                          modified_rig: Dict[str, Any], output_path: str) -> None:
    """
    Save modified session and rig metadata to output files.
    
    Args:
        session_name: Name of the session (for file naming)
        modified_session: Modified session metadata
        modified_rig: Modified rig metadata
        output_path: Base output directory path
    """
    # Create session-specific output folder with the output_path, session_name, and metadata folder
    base_output_folder = os.path.join(output_path, session_name)
    session_output_folder = os.path.join(base_output_folder, "metadata")
    os.makedirs(session_output_folder, exist_ok=True)
    
    # Save modified session metadata
    if modified_session is not None:
        output_session_json_path = os.path.join(session_output_folder, "session.json")
        with open(output_session_json_path, 'w') as f:
            json.dump(modified_session, f, indent=4)
        print(f"Saved modified session.json to: {output_session_json_path}")
    
    # Save modified rig metadata
    if modified_rig is not None:
        output_rig_json_path = os.path.join(session_output_folder, "rig.json")
        with open(output_rig_json_path, 'w') as f:
            json.dump(modified_rig, f, indent=4)
        print(f"Saved modified rig.json to: {output_rig_json_path}")

def process_single_session_metadata(session_name: str, base_path: str, output_path: str) -> bool:
    """
    Process metadata for a single session: load, modify, and save.
    
    Args:
        session_name: Name of the session to process
        base_path: Base path containing session folders
        output_path: Output directory for modified files
        
    Returns:
        True if successful, False otherwise
    """
    session_folder = os.path.join(base_path, session_name)
    
    if not os.path.exists(session_folder):
        print(f"Session folder not found: {session_folder}")
        return False
    
    print(f"\nProcessing metadata for session: {session_name}")
    
    # Load original metadata files
    session_metadata, rig_metadata = load_metadata_files(session_folder)
    
    if session_metadata is None and rig_metadata is None:
        print(f"No metadata files found for session: {session_name}")
        return False
    
    # Apply modifications
    modified_session = None
    modified_rig = None
    
    if session_metadata is not None:
        modified_session = modify_session_json(session_metadata)
        print("Applied session.json modifications")
    
    if rig_metadata is not None:
        modified_rig = modify_rig_json(rig_metadata)
        print("Applied rig.json modifications")
    
    # Save modified files
    save_modified_metadata(session_name, modified_session, modified_rig, output_path)
    
    return True

In [30]:
import pandas as pd 

# Import the recording session summary table 
recording_summary = "/Volumes/scratch/andrew.shelton/NPUltra_data/raw_npultra_data/NPUltra_recording_summary.xlsx"
recording_summary_table = pd.read_excel(recording_summary)

# Filter table for sessions of interest  
filtered_sessions = recording_summary_table[
    (recording_summary_table['experiment'] == 'NPUltra_psychedelics') &
    (recording_summary_table['uploaded to CO'] == 'yes')]

# Iterate through each session folder on VAST for sessions of interest
filtered_session_list = filtered_sessions['session'].tolist()

for idx in range(0,12): 
    output_path = '/Volumes/scratch/suyee.lee'
    session_name = filtered_session_list[idx] 

    process_single_session_metadata(
        session_name = session_name,
        base_path = '/Volumes/scratch/andrew.shelton/NPUltra_data/raw_npultra_data/',
        output_path = output_path
        ) 



Processing metadata for session: 2024-05-14_714527
Loaded session.json from: /Volumes/scratch/andrew.shelton/NPUltra_data/raw_npultra_data/2024-05-14_714527/metadata/session.json
Loaded rig.json from: /Volumes/scratch/andrew.shelton/NPUltra_data/raw_npultra_data/2024-05-14_714527/metadata/rig.json
Applied session.json modifications
Applied rig.json modifications
Saved modified session.json to: /Volumes/scratch/suyee.lee/2024-05-14_714527/metadata/session.json
Saved modified rig.json to: /Volumes/scratch/suyee.lee/2024-05-14_714527/metadata/rig.json

Processing metadata for session: 2024-05-15_714527
Loaded session.json from: /Volumes/scratch/andrew.shelton/NPUltra_data/raw_npultra_data/2024-05-15_714527/metadata/session.json
Loaded rig.json from: /Volumes/scratch/andrew.shelton/NPUltra_data/raw_npultra_data/2024-05-15_714527/metadata/rig.json
Applied session.json modifications
Applied rig.json modifications
Saved modified session.json to: /Volumes/scratch/suyee.lee/2024-05-15_714527/m

In [23]:
len(filtered_session_list)

12